# TrustSafe-RAG — Exploração das consultas (180 registros)

Notebook de exploração do CSV oficial e conferência da auditoria de escopo.
A auditoria v06 será carregada para conferir as 180 consultas (Q001 a Q180). As decisões e justificativas ficam pendentes de revisão humana; o notebook não as inventa.


## 1. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports e carregamento do dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJETO = Path("/content/drive/MyDrive/TrustSafe-RAG")
ARQUIVO_CSV = PROJETO / "data" / "raw" / "consultas_mental_v04.csv"
PASTA_FIGURAS = PROJETO / "figures"
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO_CSV.is_file():
    raise FileNotFoundError(f"CSV oficial não encontrado: {ARQUIVO_CSV}")

df = pd.read_csv(ARQUIVO_CSV, encoding="utf-8-sig")
print("Arquivo:", ARQUIVO_CSV)
print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())
display(df.head(10))

## 3. Validar estrutura e distribuição

In [ ]:
obrigatorias = [
    "id", "consulta", "risco", "justificativa",
    "comportamento_esperado", "tipo_fonte_desejavel"
]
faltantes = [c for c in obrigatorias if c not in df.columns]
assert not faltantes, f"Colunas ausentes: {faltantes}"

print("Total de consultas:", len(df))
print("IDs duplicados:", df["id"].duplicated().sum())
print("\\nValores ausentes:")
print(df[obrigatorias].isna().sum())
print("\\nDistribuição por risco:")
contagem_risco = df["risco"].value_counts().reindex(
    ["baixo", "moderado", "alto"], fill_value=0
)
print(contagem_risco)

# Valida o conjunto esperado: 180 consultas, sendo 60 por classe.
assert len(df) == 180, f"Esperadas 180 consultas, mas foram encontradas {len(df)}."
assert df["id"].is_unique, "Existem IDs duplicados."
assert (
    set(df["risco"].dropna().unique()) == {"baixo", "moderado", "alto"}
), "Há classificações de risco inesperadas."
assert contagem_risco.to_dict() == {
    "baixo": 60, "moderado": 60, "alto": 60
}, f"Distribuição diferente da esperada: {contagem_risco.to_dict()}"

# Verifica campos essenciais vazios ou preenchidos apenas com espaços.
for coluna in ["id", "consulta", "risco"]:
    vazios = df[coluna].astype(str).str.strip().eq("").sum()
    assert vazios == 0, f"A coluna '{coluna}' possui {vazios} valores vazios."

print("\\nValidação estrutural concluída: 180 consultas, 60 por classe.")

## 4. Criar ou conferir a auditoria de escopo (180 consultas)

Esta seção carrega a auditoria v06 já existente. O notebook não cria uma auditoria vazia nem altera as decisões e justificativas registradas.

In [ ]:
# Auditoria de escopo: uma linha para cada consulta do dataset atual.
PASTA_DOCS = PROJETO / "docs"
PASTA_DOCS.mkdir(parents=True, exist_ok=True)
ARQUIVO_AUDITORIA = PASTA_DOCS / "auditoria_escopo_v06.csv"

colunas_auditoria = [
    "id_consulta",
    "consulta",
    "risco",
    "tema",
    "pertence_saude_mental",
    "decisao",
    "justificativa",
    "status_revisao",
]

if not ARQUIVO_AUDITORIA.is_file():
    raise FileNotFoundError(
        f"Arquivo de auditoria v06 não encontrado: {ARQUIVO_AUDITORIA}. "
        "Confira se auditoria_escopo_v06.csv está na pasta docs."
    )

# Carrega a auditoria existente sem criar uma versão vazia ou sobrescrever dados.
auditoria = pd.read_csv(ARQUIVO_AUDITORIA, encoding="utf-8-sig")
print("Auditoria v06 carregada:", ARQUIVO_AUDITORIA)

faltam_auditoria = [c for c in colunas_auditoria if c not in auditoria.columns]
assert not faltam_auditoria, f"Colunas ausentes na auditoria: {faltam_auditoria}"

# A auditoria deve ter exatamente uma linha para cada ID do dataset atual.
assert len(auditoria) == 180, (
    f"A auditoria deve ter 180 registros; foram encontrados {len(auditoria)}. "
    "Confira auditoria_escopo_v06.csv antes de continuar."
)
assert auditoria["id_consulta"].notna().all(), "Há IDs vazios na auditoria."
assert auditoria["id_consulta"].is_unique, "Há IDs duplicados na auditoria."

ids_auditados = set(auditoria["id_consulta"].astype(str))
ids_dataset = set(df["id"].astype(str))
assert ids_auditados == ids_dataset, (
    "Os IDs da auditoria não correspondem exatamente aos 180 IDs do dataset."
)

print("\\n===== VALIDAÇÃO DA AUDITORIA =====")
print("Registros na auditoria:", len(auditoria))
print("IDs auditados:", auditoria["id_consulta"].nunique())
print("Primeiro ID:", auditoria["id_consulta"].iloc[0])
print("Último ID:", auditoria["id_consulta"].iloc[-1])
print("\\nStatus da revisão:")
print(auditoria["status_revisao"].value_counts(dropna=False))

pendentes = auditoria["status_revisao"].astype(str).str.upper().eq("PENDENTE").sum()
print("\\nRegistros ainda pendentes de revisão humana:", pendentes)
print("Auditoria estrutural: 180 linhas, uma para cada consulta.")
print("Observação: 180 linhas não significam 180 decisões já revisadas.")


## 5. Gráfico: distribuição das classes

In [ ]:
ordem = ["baixo", "moderado", "alto"]
contagem = df["risco"].value_counts().reindex(ordem, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(contagem.index, contagem.values)
ax.set_title("Distribuição das classes de risco")
ax.set_xlabel("Classe de risco")
ax.set_ylabel("Número de consultas")
ax.bar_label(ax.containers[0])
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "distribuicao_classes.png", dpi=150)
plt.show()
plt.close(fig)

## 6. Gráfico: comprimento das consultas

In [ ]:
# Calcular o número de palavras de cada consulta
df["comprimento_palavras"] = df["consulta"].astype(str).apply(
    lambda texto: len(texto.split())
)

# Gerar o histograma para as 180 consultas
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df["comprimento_palavras"], bins=15, edgecolor="black")

ax.set_title("Distribuição do comprimento das 180 consultas")
ax.set_xlabel("Número de palavras")
ax.set_ylabel("Quantidade de consultas")
fig.tight_layout()

caminho_figura = PASTA_FIGURAS / "comprimento_consultas.png"
fig.savefig(caminho_figura, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Gráfico salvo em: {caminho_figura}")

## 7. Gráfico: tipos de fontes desejáveis

In [ ]:
fontes = (
    df["tipo_fonte_desejavel"].astype(str).str.strip().value_counts().sort_values()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(fontes))))
ax.barh(fontes.index, fontes.values)
ax.set_title("Tipos de fontes desejáveis")
ax.set_xlabel("Número de consultas")
ax.set_ylabel("Tipo de fonte")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "tipos_fontes_desejaveis.png", dpi=150)
plt.show()
plt.close(fig)

## 8. Resumo final

In [ ]:
print("===== RESUMO DA EXPLORAÇÃO =====")
print("Consultas analisadas:", len(df))
print("IDs únicos:", df["id"].nunique())
print("Valores ausentes no dataset:", int(df.isna().sum().sum()))

print("\\nDistribuição por risco:")
print(df["risco"].value_counts().reindex(
    ["baixo", "moderado", "alto"], fill_value=0
))

print("\\nComprimento médio das consultas (palavras):",
      round(df["comprimento_palavras"].mean(), 2))
print("Figuras salvas em:", PASTA_FIGURAS)

## Observação

Este notebook lê o CSV oficial e recria as figuras na pasta `figures`. Na seção de auditoria, carrega `docs/auditoria_escopo_v06.csv`, com 180 linhas vinculadas aos IDs Q001–Q180.

Os campos de pertinência, decisão e justificativa começam como `PENDENTE` e precisam ser preenchidos por revisão humana. A auditoria antiga `auditoria_escopo_v02.csv` não é sobrescrita.